In [ ]:
# Install and verify packages
!pip install -q ucimlrepo imbalanced-learn xgboost lightgbm shap lime dice-ml


import importlib
for pkg in ['ucimlrepo', 'shap', 'lime', 'dice_ml', 'xgboost', 'lightgbm', 'imblearn']:
    try:
        importlib.import_module(pkg)
        print(f'  OK  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg} — re-run this cell')

print('\nAll packages ready.')


In [ ]:
# Output storage
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/HCV_XAI_Paper'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output folder ready: {OUTPUT_DIR}')


In [ ]:
# Imports and global settings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.preprocessing import label_binarize

from ucimlrepo import fetch_ucirepo

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

import shap
import lime
import lime.lime_tabular
import dice_ml
from dice_ml import Dice

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.dpi': 150,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})


LABEL_MAP = {
    '0=Blood Donor':          0,
    '0s=suspect Blood Donor': 0,
    '1=Hepatitis':            1,
    '2=Fibrosis':             2,
    '3=Cirrhosis':            3,
}
CLASS_NAMES  = ['Donor/Control', 'Hepatitis', 'Fibrosis', 'Cirrhosis']


IDX_DONOR_CONTROL = 0
IDX_HEPATITIS   = 1
IDX_FIBROSIS    = 2
IDX_CIRRHOSIS   = 3

N_KERNEL_BG  = 30
N_SHAP_EVAL  = 50

print('Imports complete.')
print('Classes:', CLASS_NAMES)


In [ ]:
# Load UCI HCV dataset
print('Loading UCI HCV dataset (id=571)...')
hcv    = fetch_ucirepo(id=571)
X_raw  = hcv.data.features.copy()
y_raw  = hcv.data.targets.copy()
df     = pd.concat([X_raw, y_raw], axis=1)


if 'X' in df.columns:
    df.drop(columns=['X'], inplace=True)

print(f'Shape          : {df.shape}')
print(f'Columns        : {list(df.columns)}')
print(f'\nClass distribution:')
print(df['Category'].value_counts())
print(f'\nMissing values:')
print(df.isnull().sum())
df.head()


In [ ]:
# Class distribution (EDA)
df_eda = df.copy()
df_eda['MergedCategory'] = df_eda['Category'].replace({
    '0=Blood Donor':          'Donor/Control',
    '0s=suspect Blood Donor': 'Donor/Control',
    '1=Hepatitis':            'Hepatitis',
    '2=Fibrosis':             'Fibrosis',
    '3=Cirrhosis':            'Cirrhosis',
})
class_order  = ['Donor/Control', 'Hepatitis', 'Fibrosis', 'Cirrhosis']
class_counts = df_eda['MergedCategory'].value_counts().reindex(class_order)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2196F3','#FF9800','#F44336','#9C27B0']
bars   = ax.bar(class_counts.index, class_counts.values, color=colors)
for bar, v in zip(bars, class_counts.values):
    pct = v / len(df_eda) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{v}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Disease category (merged 4-class)')
ax.set_ylabel('Number of patients')
ax.set_title('Class distribution — UCI HCV dataset (Donor/Control merge)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_class_distribution.png', bbox_inches='tight')
plt.show()
print('NOTE: Severe class imbalance remains — SMOTE applied to training data only.')


In [ ]:
# Missing-value summary
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_pct = missing_pct[missing_pct > 0]
if not missing_pct.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    missing_pct.plot(kind='bar', ax=ax, color='#FF7043')
    ax.set_title('Missing values per feature (%)')
    ax.set_ylabel('% missing')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig2_missing_values.png')
    plt.show()
else:
    print('No missing values found.')


In [ ]:
# Feature distributions by class
df_plot = df.copy()

df_plot['Category'] = df_plot['Category'].replace({
    '0=Blood Donor': 'Donor/Control', '0s=suspect Blood Donor': 'Donor/Control',
    '1=Hepatitis': 'Hepatitis', '2=Fibrosis': 'Fibrosis', '3=Cirrhosis': 'Cirrhosis',
})
if 'CGT' in df_plot.columns:
    df_plot.rename(columns={'CGT': 'GGT'}, inplace=True)
numeric_cols = df_plot.select_dtypes(include=np.number).columns.tolist()
ncols = 4
nrows = (len(numeric_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df_plot.boxplot(column=col, by='Category', ax=axes[i])
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30, labelsize=7)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature distributions by HCV category', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_eda_feature_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
# Correlation heatmap
df_plot = df.copy()
if 'CGT' in df_plot.columns:
    df_plot.rename(columns={'CGT': 'GGT'}, inplace=True)
numeric_cols = df_plot.select_dtypes(include=np.number).columns.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_plot[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_eda_correlation_heatmap.png')
plt.show()


In [ ]:
# Encode target and split features
df_proc = df.copy()


df_proc['target'] = df_proc['Category'].map(LABEL_MAP)
df_proc.dropna(subset=['target'], inplace=True)
df_proc['target'] = df_proc['target'].astype(int)
df_proc.drop(columns=['Category'], inplace=True)


df_proc['Sex'] = df_proc['Sex'].map({'f': 0, 'm': 1}).fillna(0).astype(int)
if 'CGT' in df_proc.columns:
    df_proc.rename(columns={'CGT': 'GGT'}, inplace=True)

FEATURE_NAMES = [c for c in df_proc.columns if c != 'target']
X = df_proc[FEATURE_NAMES].values
y = df_proc['target'].values

print(f'Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')
print(f'Class counts (4-class): {dict(zip(CLASS_NAMES, np.bincount(y)))}')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'\nTrain: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')


imputer    = SimpleImputer(strategy='median')
X_train    = imputer.fit_transform(X_train)
X_test     = imputer.transform(X_test)
print('Imputation complete (median, fitted on training set only).')


scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('Scaling complete (fitted on training set only).')


smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'After SMOTE — train size: {X_train_res.shape[0]}')
print(f'SMOTE class dist: {dict(zip(CLASS_NAMES, np.bincount(y_train_res)))}')
print('\nPreprocessing complete — no data leakage.')


In [ ]:
# Model definitions
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, random_state=RANDOM_STATE,
        eval_metric='mlogloss', verbosity=0, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf', probability=True, random_state=RANDOM_STATE, C=1.0
    ),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    trained_models[name] = model
    print(f'  Trained: {name}')

print('\nAll models trained.')


In [ ]:
# Hyperparameter summary table
hyperparam_rows = [
    {'Model': 'Logistic Regression',
     'Key hyperparameters': 'penalty=l2 (default), C=1.0 (default), solver=lbfgs, max_iter=2000',
     'Tuning strategy': 'Fixed / default'},
    {'Model': 'Random Forest',
     'Key hyperparameters': 'n_estimators=100, criterion=gini, max_depth=None (default)',
     'Tuning strategy': 'Fixed / default'},
    {'Model': 'XGBoost',
     'Key hyperparameters': 'n_estimators=100, learning_rate=0.3 (default), max_depth=6 (default), eval_metric=mlogloss',
     'Tuning strategy': 'Fixed / default'},
    {'Model': 'LightGBM',
     'Key hyperparameters': 'n_estimators=100, learning_rate=0.1 (default), num_leaves=31 (default)',
     'Tuning strategy': 'Fixed / default'},
    {'Model': 'SVM',
     'Key hyperparameters': 'kernel=rbf, C=1.0, gamma=scale (default), probability=True',
     'Tuning strategy': 'Fixed / default'},
]
hyperparam_df = pd.DataFrame(hyperparam_rows)
hyperparam_df.to_csv(f'{OUTPUT_DIR}/table_model_hyperparameters.csv', index=False)
print('Model hyperparameters:')
display(hyperparam_df)


In [ ]:
# 5-fold stratified cross-validation
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

N_SPLITS = 5
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_results = {name: {'acc':[], 'f1':[], 'auc':[]} for name in models}

print(f'{N_SPLITS}-Fold Stratified Cross-Validation')
print('=' * 55)

for name, model_proto in models.items():
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]


        imp        = SimpleImputer(strategy='median')
        X_tr       = imp.fit_transform(X_tr)
        X_va       = imp.transform(X_va)


        sc         = StandardScaler()
        X_tr       = sc.fit_transform(X_tr)
        X_va       = sc.transform(X_va)


        sm         = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
        X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)


        mdl        = clone(model_proto)
        mdl.fit(X_tr_res, y_tr_res)


        y_pred     = mdl.predict(X_va)
        y_prob     = mdl.predict_proba(X_va)
        cv_results[name]['acc'].append(accuracy_score(y_va, y_pred))
        cv_results[name]['f1'].append(f1_score(y_va, y_pred, average='macro', zero_division=0))
        try:
            cv_results[name]['auc'].append(
                roc_auc_score(y_va, y_prob, multi_class='ovr', average='macro'))
        except Exception:
            cv_results[name]['auc'].append(np.nan)

cv_rows = []
for name in models:
    acc_arr = np.array(cv_results[name]['acc'])
    f1_arr  = np.array(cv_results[name]['f1'])
    auc_arr = np.array(cv_results[name]['auc'])
    cv_rows.append({
        'Model':         name,
        'Accuracy':      f'{acc_arr.mean():.4f} ± {acc_arr.std():.4f}',
        'Macro F1':      f'{f1_arr.mean():.4f} ± {f1_arr.std():.4f}',
        'AUC-ROC':       f'{auc_arr.mean():.4f} ± {auc_arr.std():.4f}',
    })

cv_df = pd.DataFrame(cv_rows).set_index('Model')
cv_df.to_csv(f'{OUTPUT_DIR}/table2_cv_results.csv')
print('\nCross-Validation Results (mean ± std) — TABLE 2:')
display(cv_df)


In [ ]:
# Held-out test evaluation
from sklearn.metrics import roc_curve, auc as sklearn_auc

rows = []


fig, axes = plt.subplots(1, len(trained_models), figsize=(22, 5))
for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)

    acc      = accuracy_score(y_test, y_pred)
    f1_mac   = f1_score(y_test, y_pred, average='macro',    zero_division=0)
    f1_wt    = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    try:
        auc_v = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    except Exception:
        auc_v = np.nan

    rows.append({'Model': name, 'Accuracy': round(acc, 4),
                 'Macro F1': round(f1_mac, 4),
                 'Weighted F1': round(f1_wt, 4),
                 'AUC (OvR)': round(auc_v, 4)})

    cm = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASS_NAMES))))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(['DC','HEP','FIB','CIR'], fontsize=8)
    ax.set_yticklabels(['DC','HEP','FIB','CIR'], fontsize=8)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.set_title(name, fontsize=9, fontweight='bold')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.suptitle('Confusion matrices — all models (raw counts)', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/supp_confusion_matrices_raw.png', bbox_inches='tight')
plt.show()


fig, axes = plt.subplots(1, len(trained_models), figsize=(22, 5))
fig.suptitle('Figure 4: Confusion Matrices — All Five Models (Test Set)', fontsize=13, fontweight='bold', y=1.02)
for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred  = model.predict(X_test_scaled)
    cm      = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASS_NAMES))))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(['DC','HEP','FIB','CIR'], fontsize=8)
    ax.set_yticklabels(['DC','HEP','FIB','CIR'], fontsize=8)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    ax.set_title(name, fontsize=9, fontweight='bold')
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            color = 'white' if cm_norm[i,j] > 0.5 else '#1F4E79'
            ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                    ha='center', va='center', fontsize=7, color=color, fontweight='bold')
plt.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig4_confusion_matrices.png', bbox_inches='tight')
plt.show()


results_df = pd.DataFrame(rows).set_index('Model')
results_df.to_csv(f'{OUTPUT_DIR}/table1_model_performance.csv')
print('\nTable 1 — Model performance:')
display(results_df)


In [ ]:
# Performance comparison plot
fig, ax = plt.subplots(figsize=(12, 6))
x      = np.arange(len(results_df))
width  = 0.25
metrics  = ['Accuracy', 'Macro F1', 'AUC (OvR)']
colors   = ['#1F4E79', '#2E75B6', '#5BA3D9']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    rects = ax.bar(x + (i-1)*width, results_df[metric], width,
                   label=metric, color=color)
    for rect in rects:
        ax.text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.005,
                f'{rect.get_height():.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Figure 3: Model Performance Comparison\n(Accuracy · Macro F1 · AUC-ROC on Held-Out Test Set)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(1.0, color='#AAAAAA', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_model_performance.png', bbox_inches='tight')
plt.show()


for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    print(f'\n{"-"*55}\n{name}\n{"-"*55}')
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# Per-class ROC curves
from sklearn.metrics import roc_curve
from sklearn.metrics import auc as sklearn_auc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Figure 5: ROC Curves', fontsize=13, fontweight='bold')
roc_colors = ['#1F4E79','#E74C3C','#27AE60','#8E44AD','#F39C12']


for (name, model), color in zip(trained_models.items(), roc_colors):
    y_prob = model.predict_proba(X_test_scaled)
    y_bin  = label_binarize(y_test, classes=list(range(len(CLASS_NAMES))))
    fpr_all, tpr_all = [], []
    for ci in range(y_bin.shape[1]):
        if y_bin[:, ci].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, ci], y_prob[:, ci])
        fpr_all.append(fpr)
        tpr_all.append(tpr)
    mean_fpr = np.linspace(0, 1, 200)
    mean_tpr = np.mean([np.interp(mean_fpr, f, t) for f, t in zip(fpr_all, tpr_all)], axis=0)
    macro_auc = sklearn_auc(mean_fpr, mean_tpr)


    ax1.plot(mean_fpr, mean_tpr, color=color, lw=2, label=f'{name} (AUC={macro_auc:.3f})')

ax1.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax1.set_xlabel('False Positive Rate', fontweight='bold')
ax1.set_ylabel('True Positive Rate', fontweight='bold')
ax1.set_title('Macro-Averaged ROC Curves\n(One-vs-Rest per Model)', fontweight='bold')
ax1.legend(fontsize=8, loc='lower right')


best_model_name = results_df['Macro F1'].idxmax()
best_model      = trained_models[best_model_name]
y_prob    = best_model.predict_proba(X_test_scaled)
y_bin     = label_binarize(y_test, classes=list(range(len(CLASS_NAMES))))
cls_colors = ['#1F4E79','#2E75B6','#E74C3C','#E67E22']
for ci, (cls_name, color) in enumerate(zip(CLASS_NAMES, cls_colors)):
    if ci >= y_bin.shape[1] or y_bin[:, ci].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_bin[:, ci], y_prob[:, ci])
    ax2.plot(fpr, tpr, color=color, lw=2,
             label=f'{cls_name} (AUC={sklearn_auc(fpr, tpr):.3f})')

ax2.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax2.set_xlabel('False Positive Rate', fontweight='bold')
ax2.set_ylabel('True Positive Rate', fontweight='bold')
ax2.set_title(f'Per-Class ROC Curves — {best_model_name}\n(Best Held-Out Macro-F1 Model)', fontweight='bold')
ax2.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig5_roc_curves.png', bbox_inches='tight')
plt.show()


In [ ]:
# SHAP values and global importance
shap_values_dict = {}
shap_global_imp  = {}
shap_rankings    = {}

HEPATITIS_CLASS_IDX = IDX_HEPATITIS


X_shap_eval = X_test_scaled[:N_SHAP_EVAL]

beeswarm_fig_nums = {'Logistic Regression': 7, 'Random Forest': 8,
                     'XGBoost': 9, 'LightGBM': 10, 'SVM': 11}
for name, model in trained_models.items():
    print(f'\nComputing SHAP — {name}...')
    try:
        if name in ['Random Forest', 'XGBoost', 'LightGBM']:
            explainer = shap.TreeExplainer(model)
            sv        = explainer.shap_values(X_shap_eval)
            X_plot    = X_shap_eval
        else:
            background = shap.kmeans(X_train_scaled, N_KERNEL_BG)
            explainer  = shap.KernelExplainer(model.predict_proba, background)
            sv         = explainer.shap_values(X_shap_eval)
            X_plot     = X_shap_eval

        assert isinstance(sv, np.ndarray), f'Expected ndarray, got {type(sv)}'
        assert sv.ndim == 3, f'Expected 3D, got shape {sv.shape}'
        print(f'  sv.shape = {sv.shape}  (samples, features, classes)')

        shap_values_dict[name] = sv

        importance = np.abs(sv).mean(axis=(0, 2))
        shap_global_imp[name]  = importance
        shap_rankings[name]    = [FEATURE_NAMES[i] for i in np.argsort(importance)[::-1]]
        print(f'  Top-5: {shap_rankings[name][:5]}')

        sv_hepatitis = sv[:, :, HEPATITIS_CLASS_IDX]
        plt.figure(figsize=(10, 6))
        shap.summary_plot(sv_hepatitis, X_plot,
                          feature_names=FEATURE_NAMES, show=False, plot_type='dot')
        plt.title(f'SHAP beeswarm — {name}  (class: Hepatitis)', fontsize=12, pad=12)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/fig{beeswarm_fig_nums[name]}_shap_beeswarm_{name.replace(" ","_")}.png',
            dpi=150, bbox_inches='tight')
        plt.show()

        feat_imp = pd.Series(importance, index=FEATURE_NAMES).sort_values(ascending=True)
        fig, ax  = plt.subplots(figsize=(8, 5))
        feat_imp.plot(kind='barh', ax=ax, color='#1976D2')
        ax.set_xlabel('Mean |SHAP value| (avg over samples & classes)')
        ax.set_title(f'SHAP feature importance — {name}')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/fig_shap_bar_{name.replace(" ","_")}.png', dpi=150)
        plt.show()

    except Exception as e:
        import traceback
        print(f'  ERROR for {name}:')
        traceback.print_exc()
        shap_rankings[name]   = FEATURE_NAMES[:]
        shap_global_imp[name] = np.zeros(len(FEATURE_NAMES))

print('\nSHAP analysis complete.')


In [ ]:
# Global SHAP importance plot
fig, axes = plt.subplots(1, len(trained_models), figsize=(22, 6), sharey=False)
fig.suptitle('Figure 6: Global SHAP Feature Importance — All Five Models',
             fontsize=13, fontweight='bold', y=1.02)
bar_colors = ['#1F4E79','#2E75B6','#27AE60','#8E44AD','#E74C3C']

for ax, (name, model), color in zip(axes, trained_models.items(), bar_colors):
    imp  = pd.Series(shap_global_imp[name], index=FEATURE_NAMES).sort_values(ascending=True)
    bars = ax.barh(imp.index, imp.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Mean |SHAP|', fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

    max_val = imp.max()
    for bar in bars:
        if abs(bar.get_width() - max_val) < 1e-10:
            bar.set_edgecolor('#FFD700')
            bar.set_linewidth(2.5)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig6_shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# LIME explainer setup
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_train_res,
    feature_names = FEATURE_NAMES,
    class_names   = CLASS_NAMES,
    mode          = 'classification',
    random_state  = RANDOM_STATE,
)


representative_instances = {}
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = (y_test == cls_idx)
    if mask.sum() == 0:
        print(f'  WARNING: No {cls_name} instances in test set — skipping')
        continue
    cls_pts  = X_test_scaled[mask]
    centroid = cls_pts.mean(axis=0)
    dists    = np.linalg.norm(cls_pts - centroid, axis=1)
    representative_instances[cls_name] = cls_pts[np.argmin(dists)]

print('Representative instances selected:')
for cls_name in representative_instances:
    print(f'  {cls_name}')

lime_rankings = {name: {} for name in trained_models}


In [ ]:
# LIME local explanations per model
lime_fig_nums = {'Logistic Regression': 14, 'Random Forest': 15,
                 'XGBoost': 16, 'LightGBM': 17, 'SVM': 18}
for model_name, model in trained_models.items():
    print(f'\nLIME — {model_name}')
    n_inst = len(representative_instances)
    fig, axes = plt.subplots(1, n_inst, figsize=(5 * n_inst, 5))
    if n_inst == 1:
        axes = [axes]

    for ax, (cls_name, instance) in zip(axes, representative_instances.items()):
        try:


            cls_idx_lime = CLASS_NAMES.index(cls_name)

            exp = lime_explainer.explain_instance(
                data_row   = instance,
                predict_fn = model.predict_proba,
                num_features = len(FEATURE_NAMES),
                num_samples  = 5000,
                labels     = [cls_idx_lime],
            )
            exp_map  = exp.as_map()[cls_idx_lime]
            feat_lbl = [FEATURE_NAMES[i] for i, _ in exp_map]
            weights  = [w for _, w in exp_map]


            lime_rankings[model_name][cls_name] = [
                f for f, _ in sorted(zip(feat_lbl, weights),
                                     key=lambda x: abs(x[1]), reverse=True)
            ]

            colors = ['#4CAF50' if w > 0 else '#F44336' for w in weights]
            ax.barh(feat_lbl, weights, color=colors)
            ax.axvline(0, color='black', linewidth=0.8)
            ax.set_title(cls_name, fontsize=9)
            ax.set_xlabel('LIME weight', fontsize=8)

        except Exception as e:
            ax.set_title(f'{cls_name}\n(failed)', fontsize=8)
            print(f'  WARNING: {cls_name} — {e}')

    plt.suptitle(f'LIME local explanations — {model_name}')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig{lime_fig_nums[model_name]}_lime_{model_name.replace(" ","_")}.png',
            dpi=150, bbox_inches='tight')
    plt.show()

print('\nLIME analysis complete.')


In [ ]:
# DiCE data and model setup
X_train_df           = pd.DataFrame(X_train_res, columns=FEATURE_NAMES)
X_train_df['target'] = y_train_res
X_test_df            = pd.DataFrame(X_test_scaled, columns=FEATURE_NAMES)


dice_model_name = 'LightGBM'
dice_ml_model   = trained_models[dice_model_name]

d   = dice_ml.Data(dataframe=X_train_df,
                   continuous_features=FEATURE_NAMES,
                   outcome_name='target')
m   = dice_ml.Model(model=dice_ml_model, backend='sklearn', model_type='classifier')
exp = Dice(d, m, method='random')


IMMUTABLE_FEATURES = [f for f in ['Age', 'Sex'] if f in FEATURE_NAMES]
MUTABLE_FEATURES   = [f for f in FEATURE_NAMES if f not in IMMUTABLE_FEATURES]

print(f'DiCE configured with : {dice_model_name}')
print(f'Immutable (fixed)    : {IMMUTABLE_FEATURES}')
print(f'Mutable (can vary)   : {MUTABLE_FEATURES}')
print(f'Test class distribution: {dict(zip(CLASS_NAMES, np.bincount(y_test)))}')


In [ ]:
# DiCE counterfactual generation
cirr_idx = np.where(y_test == IDX_CIRRHOSIS)[0]
fibr_idx = np.where(y_test == IDX_FIBROSIS)[0]

print('=' * 65)
print('DiCE counterfactual values (Age and Sex fixed):')
print('=' * 65)

cf_results_table = []

for label, cls_idx, desired_cls, desired_name in [
    ('Patient A (Cirrhosis)', cirr_idx, IDX_DONOR_CONTROL, 'Donor/Control'),
    ('Patient B (Cirrhosis)', cirr_idx, IDX_HEPATITIS,   'Hepatitis'),
    ('Patient C (Fibrosis)',  fibr_idx, IDX_DONOR_CONTROL, 'Donor/Control'),
]:
    if len(cls_idx) == 0:
        print(f'\n{label}: No instances in test set')
        continue

    query = X_test_df.iloc[cls_idx[0]:cls_idx[0]+1]

    try:

        permitted = {feat: [-3.5, 3.5] for feat in MUTABLE_FEATURES}
        cf = exp.generate_counterfactuals(
            query,
            total_CFs          = 4,
            desired_class      = desired_cls,
            features_to_vary   = MUTABLE_FEATURES,
            permitted_range    = permitted
        )
        cf_df   = cf.cf_examples_list[0].final_cfs_df[FEATURE_NAMES]
        orig    = query[FEATURE_NAMES].values[0]
        cf_vals = cf_df.values[0]

        print(f'\n{label}  ->  Target stage: {desired_name}')
        print(f'{"Feature":<8}  {"Original":>10}  {"Counterfactual":>15}  {"Change":>10}')
        print('-' * 52)
        for feat, o, c in zip(FEATURE_NAMES, orig, cf_vals):
            if abs(o - c) > 0.01:
                print(f'{feat:<8}  {o:>10.3f}  {c:>15.3f}  {c-o:>+10.3f}')
                cf_results_table.append({
                    'Patient': label, 'Target': desired_name,
                    'Feature': feat, 'Original': round(o, 3),
                    'Counterfactual': round(c, 3), 'Change': round(c-o, 3)
                })

    except Exception as e:
        print(f'\n{label}: ERROR — {e}')

print('\n' + '=' * 65)
print('NOTE: standardised units (mean=0, std=1). Age & Sex held fixed.')
print('=' * 65)

if cf_results_table:
    cf_df_out = pd.DataFrame(cf_results_table)
    cf_df_out.to_csv(f'{OUTPUT_DIR}/table4_counterfactuals.csv', index=False)
    print('\nTABLE 4 saved to Drive.')
    display(cf_df_out)


In [ ]:
# DiCE counterfactual plot
if len(cirr_idx) > 0:
    query = X_test_df.iloc[cirr_idx[0]:cirr_idx[0]+1]
    try:
        permitted = {feat: [-3.5, 3.5] for feat in MUTABLE_FEATURES}
        cf = exp.generate_counterfactuals(
            query,
            total_CFs        = 4,
            desired_class    = IDX_DONOR_CONTROL,
            features_to_vary = MUTABLE_FEATURES,
            permitted_range  = permitted
        )
        cf_df = cf.cf_examples_list[0].final_cfs_df[FEATURE_NAMES]

        top6_idx   = np.argsort(shap_global_imp['LightGBM'])[::-1][:6]
        top6_feats = [FEATURE_NAMES[i] for i in top6_idx]

        orig_row = query.copy()
        orig_row['type'] = 'Original (Cirrhosis)'
        cf_df = cf_df.copy()
        cf_df['type'] = [f'CF {i+1}' for i in range(len(cf_df))]
        plot_df = pd.concat([orig_row, cf_df], ignore_index=True)

        fig, ax = plt.subplots(figsize=(13, 6))
        cf_colors = ['#E74C3C','#2E75B6','#27AE60','#8E44AD','#F39C12']
        for ri, (_, row) in enumerate(plot_df.iterrows()):
            vals = [row[f] for f in top6_feats]
            ax.plot(range(len(top6_feats)), vals,
                    color=cf_colors[ri % len(cf_colors)],
                    lw=2.5 if ri == 0 else 1.8,
                    linestyle='-' if ri == 0 else '--',
                    marker='o', markersize=7,
                    label=row['type'], alpha=0.9)

        ax.set_xticks(range(len(top6_feats)))
        ax.set_xticklabels(top6_feats, fontsize=12, fontweight='bold')
        ax.set_ylabel('Standardised Feature Value', fontsize=11, fontweight='bold')
        ax.set_title(
            'Figure 20: DiCE Counterfactual Explanations — Cirrhosis Patient\n'
            '(Feature changes required to achieve Donor/Control prediction; Age & Sex fixed)',
            fontsize=11, fontweight='bold', pad=15)
        ax.legend(fontsize=10, loc='upper right')
        ax.axhline(0, color='#AAAAAA', linewidth=0.8, linestyle=':')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/fig20_dice_counterfactuals.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('DiCE parallel coordinates plot saved.')

    except Exception as e:
        import traceback
        print(f'DiCE plot error: {e}')
        traceback.print_exc()
else:
    print('No Cirrhosis instances in test set — DiCE plot skipped.')


In [ ]:
# Cross-model SHAP Spearman agreement
model_names = list(shap_global_imp.keys())
n = len(model_names)
rho_matrix = np.zeros((n, n))

for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        rho, _ = spearmanr(shap_global_imp[m1], shap_global_imp[m2])
        rho_matrix[i, j] = round(rho, 3)

agreement_df = pd.DataFrame(rho_matrix,
                             index=model_names,
                             columns=model_names)


short_names = ['LR', 'RF', 'XGBoost', 'LightGBM', 'SVM']
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(rho_matrix, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=10)
ax.set_yticklabels(model_names, fontsize=10)
for i in range(n):
    for j in range(n):
        color = 'white' if rho_matrix[i,j] > 0.7 else '#1F4E79'
        ax.text(j, i, f'{rho_matrix[i,j]:.3f}',
                ha='center', va='center', fontsize=11, fontweight='bold', color=color)
plt.colorbar(im, ax=ax, label='Spearman rho', fraction=0.046, pad=0.04)
ax.set_title(
    'Figure 13: Cross-Model SHAP Agreement\n'
    '(Spearman Rank Correlation of Global Feature Importance)',
    fontsize=11, fontweight='bold', pad=15)
ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('Model', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig13_shap_spearman_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

agreement_df.to_csv(f'{OUTPUT_DIR}/supp_shap_spearman_agreement.csv')
print('Cross-model SHAP agreement matrix:')
display(agreement_df)


In [ ]:
# Tree-model SHAP agreement heatmap
tree_model_names = ['Random Forest', 'XGBoost', 'LightGBM']
tree_sv_dict     = {k: shap_values_dict[k] for k in tree_model_names
                    if k in shap_values_dict}

n_cls = len(CLASS_NAMES)
n_mod = len(tree_sv_dict)

if n_mod > 0:
    fig, axes = plt.subplots(n_cls, n_mod,
                             figsize=(5 * n_mod, 3.5 * n_cls))
    axes = np.array(axes).reshape(n_cls, n_mod)

    for ci, cls_name in enumerate(CLASS_NAMES):
        for mi, (mname, sv) in enumerate(tree_sv_dict.items()):
            ax = axes[ci][mi]

            mean_abs = np.abs(sv[:, :, ci]).mean(axis=0)
            feat_s   = pd.Series(mean_abs, index=FEATURE_NAMES)\
                         .sort_values(ascending=True).tail(8)
            feat_s.plot(kind='barh', ax=ax, color='#1565C0')
            ax.set_title(f'{cls_name}\n{mname}', fontsize=8)
            ax.set_xlabel('Mean |SHAP|', fontsize=8)

    plt.suptitle('Stage-specific SHAP feature importance (top 8)',
                 y=1.01, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/fig12_stage_specific_shap.png',
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No tree-based SHAP values available.')


In [ ]:
# SHAP-LIME top-5 overlap
K = 5
overlap_matrix = np.zeros((len(trained_models), len(CLASS_NAMES)))
rows_sl        = []

for mi, model_name in enumerate(trained_models.keys()):
    shap_top = set(shap_rankings[model_name][:K])
    for ci, cls_name in enumerate(CLASS_NAMES):
        lime_top = set(lime_rankings.get(model_name, {}).get(cls_name, [])[:K])
        overlap  = len(shap_top & lime_top)
        overlap_matrix[mi, ci] = overlap
        rows_sl.append({
            'Model':   model_name,
            'Class':   cls_name,
            f'Top-{K} overlap': overlap,
            'SHAP top features': ', '.join(shap_rankings[model_name][:K]),
            'LIME top features': ', '.join(lime_rankings.get(model_name, {}).get(cls_name, [])[:K]),
        })

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(overlap_matrix, cmap='YlOrRd', vmin=0, vmax=5, aspect='auto')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(trained_models)))
ax.set_xticklabels(['Donor/\nControl','Hepatitis','Fibrosis','Cirrhosis'], fontsize=10)
ax.set_yticklabels(list(trained_models.keys()), fontsize=10)
for i in range(overlap_matrix.shape[0]):
    for j in range(overlap_matrix.shape[1]):
        val   = int(overlap_matrix[i, j])
        color = 'white' if val >= 4 else '#1F4E79'
        ax.text(j, i, f'{val}/{K}',
                ha='center', va='center', fontsize=13, fontweight='bold', color=color)
plt.colorbar(im, ax=ax, label=f'Features in common (out of {K})',
             fraction=0.046, pad=0.04, ticks=[0,1,2,3,4,5])
ax.set_title(
    f'Figure 19: SHAP-LIME Top-{K} Feature Overlap\n'
    '(Number of Features in Common per Model x Class)',
    fontsize=11, fontweight='bold', pad=15)
ax.set_xlabel('Target Class', fontweight='bold')
ax.set_ylabel('Model', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig19_shap_lime_overlap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

sl_df = pd.DataFrame(rows_sl)
sl_df.to_csv(f'{OUTPUT_DIR}/table3_shap_lime_agreement.csv', index=False)

mean_overlap = sl_df.groupby('Model')[f'Top-{K} overlap'].mean().round(2)
print('TABLE 3 — SHAP vs LIME agreement:')
display(sl_df)
print('\nPer-model mean overlap (out of 5):')
display(mean_overlap)


In [ ]:
# Runtime of XAI methods
import time

runtime_rows = []


for name, model in trained_models.items():
    t0 = time.time()
    if name in ['Random Forest', 'XGBoost', 'LightGBM']:
        _expl = shap.TreeExplainer(model)
        _     = _expl.shap_values(X_shap_eval)
    else:
        _bg   = shap.kmeans(X_train_scaled, N_KERNEL_BG)
        _expl = shap.KernelExplainer(model.predict_proba, _bg)
        _     = _expl.shap_values(X_shap_eval)
    runtime_rows.append({'Method': 'SHAP', 'Model': name,
                         'Scope': f'{N_SHAP_EVAL} instances',
                         'Runtime (s)': round(time.time() - t0, 2)})


t0 = time.time()
_ = lime_explainer.explain_instance(
        data_row     = list(representative_instances.values())[0],
        predict_fn   = trained_models['LightGBM'].predict_proba,
        num_features = len(FEATURE_NAMES),
        num_samples  = 5000,
)
runtime_rows.append({'Method': 'LIME', 'Model': 'LightGBM',
                     'Scope': '1 instance, 5000 samples',
                     'Runtime (s)': round(time.time() - t0, 2)})


t0 = time.time()
_ = exp.generate_counterfactuals(
        X_test_df.iloc[cirr_idx[0]:cirr_idx[0]+1],
        total_CFs        = 4,
        desired_class    = IDX_DONOR_CONTROL,
        features_to_vary = MUTABLE_FEATURES,
        permitted_range  = {f: [-3.5, 3.5] for f in MUTABLE_FEATURES},
)
runtime_rows.append({'Method': 'DiCE', 'Model': 'LightGBM',
                     'Scope': '1 query, 4 counterfactuals',
                     'Runtime (s)': round(time.time() - t0, 2)})

runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(f'{OUTPUT_DIR}/table_runtime.csv', index=False)
print('Computational cost of XAI methods:')
display(runtime_df)


In [ ]:
# Repeated-seed CV stability
from sklearn.base import clone

REPEAT_SEEDS = [42, 7, 123, 2024, 99]
repeat_f1 = {name: [] for name in models}

for seed in REPEAT_SEEDS:
    skf_r = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for name, proto in models.items():
        fold_f1 = []
        for tr_idx, va_idx in skf_r.split(X, y):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]
            imp = SimpleImputer(strategy='median'); X_tr = imp.fit_transform(X_tr); X_va = imp.transform(X_va)
            sc  = StandardScaler();                 X_tr = sc.fit_transform(X_tr);  X_va = sc.transform(X_va)
            sm  = SMOTE(random_state=seed, k_neighbors=5)
            X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)
            mdl = clone(proto); mdl.fit(X_tr_res, y_tr_res)
            fold_f1.append(f1_score(y_va, mdl.predict(X_va), average='macro', zero_division=0))
        repeat_f1[name].append(np.mean(fold_f1))

stab_rows = []
for name in models:
    arr = np.array(repeat_f1[name])
    stab_rows.append({'Model': name,
                      'Mean macro-F1': round(arr.mean(), 4),
                      'Std across seeds': round(arr.std(), 4),
                      'Min': round(arr.min(), 4),
                      'Max': round(arr.max(), 4)})
stability_df = pd.DataFrame(stab_rows).set_index('Model')
stability_df.to_csv(f'{OUTPUT_DIR}/table_cv_stability.csv')
print('Repeated-CV stability — macro-F1 across seeds', REPEAT_SEEDS, ':')
display(stability_df)


In [ ]:
# List saved output files
import os
files = sorted(os.listdir(OUTPUT_DIR))
print(f'All outputs saved to: {OUTPUT_DIR}\n')
print(f'{"File":<60} {"Size":>10}')
print('-' * 72)
total_bytes = 0
for f in files:
    fp   = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fp)
    total_bytes += size
    print(f'{f:<60} {size:>8,} B')
print('-' * 72)
print(f'{"TOTAL":<60} {total_bytes:>8,} B  ({total_bytes/1024/1024:.1f} MB)')
print(f'\nTotal files: {len(files)}')
print('\nPipeline complete.')